# 🏀 バスケ試合動画の選手を「色分け＆追跡」する（YOLO + SAM2）

note記事「[バスケの試合動画をAIに見せたら、選手が勝手に色分け＆追跡された](https://note.com/hirosuke_0520/n/n85d57196fffd)」の内容を
**Google Colab の無料GPU** で再現するノートブックです。

パイプラインの流れ:

1. **選手検出** … YOLO11 で人物を検出
2. **ID追跡** … ByteTrack で各選手に安定したトラッキングIDを付与
3. **チーム色分け** … 各選手のユニフォーム色を抽出し、KMeans で2チームにクラスタリング → チームごとに色付きの枠を描画
4. **注釈付き動画の書き出し** … 色分けした動画を出力
5. **個人の精密追跡（SAM2）** … 1人の選手を選んで SAM2 のビデオ予測器でマスク追跡

> 💡 **使い方**: 上のメニューから「ランタイム → ランタイムのタイプを変更 → **GPU (T4)**」を選んでから、
> 上のセルから順番に実行してください。バスケの試合動画（mp4）を1本用意しておくとスムーズです。


## 1. 環境セットアップ

必要なライブラリをインストールします（初回は数分かかります）。

In [ ]:
# GPUの確認（"Tesla T4" などと出ればOK。何も出ない場合はランタイムをGPUに変更）
!nvidia-smi -L || echo "⚠️ GPUが見つかりません。ランタイム → ランタイムのタイプを変更 → GPU を選択してください。"


In [ ]:
# 依存ライブラリのインストール
#   ultralytics : YOLO11 + SAM2 をまとめて提供
#   supervision : 検出結果の描画・トラッキング補助
%pip install -q "ultralytics>=8.3.0" "supervision>=0.24.0" scikit-learn
print("✅ インストール完了")


In [ ]:
import cv2
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict, Counter

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else "cpu"


## 2. 解析する動画を用意する

以下のいずれかで動画を用意します。

- **A. 自分の動画をアップロード**: 下のセルを実行してファイルを選択
- **B. サンプル動画で試す**: `USE_SAMPLE = True` にして実行（短いフリー素材を取得）

数十秒〜1分程度の短いクリップから始めるのがおすすめです（長いと処理に時間がかかります）。


In [ ]:
USE_SAMPLE = False  # サンプル動画を使う場合は True

VIDEO_PATH = None
if USE_SAMPLE:
    # フリーのサンプル動画（バスケ）を取得。取得できない場合は手動アップロードに切替
    import urllib.request
    sample_url = "https://media.roboflow.com/supervision/video-examples/basketball-1.mp4"
    VIDEO_PATH = "input.mp4"
    try:
        urllib.request.urlretrieve(sample_url, VIDEO_PATH)
        print("✅ サンプル動画を取得:", VIDEO_PATH)
    except Exception as e:
        print("⚠️ サンプル取得に失敗しました。手動アップロードに切り替えます:", e)
        USE_SAMPLE = False

if not USE_SAMPLE:
    from google.colab import files
    print("動画ファイル(mp4)を選択してください…")
    uploaded = files.upload()
    VIDEO_PATH = list(uploaded.keys())[0]
    print("✅ アップロード:", VIDEO_PATH)

# 動画情報の表示
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"📹 {VIDEO_PATH}: {w}x{h}, {fps:.1f}fps, {total}フレーム (~{total/max(fps,1):.1f}秒)")


## 3. 選手検出 + ID追跡（YOLO11 + ByteTrack）

YOLO11 で人物（`person` クラス）を検出し、ByteTrack で各選手に一貫したIDを付けます。
モデルの重みは初回実行時に自動ダウンロードされます。


In [ ]:
from ultralytics import YOLO

# yolo11 の中サイズモデル。軽くしたいなら yolo11n.pt、精度重視なら yolo11x.pt
det_model = YOLO("yolo11m.pt")

PERSON_CLASS = 0   # COCOの 'person'
CONF = 0.35        # 検出の信頼度しきい値

# 動画全体をトラッキング。track_history にフレームごとの検出を貯める
#   track_history[track_id] = [(frame_idx, x1,y1,x2,y2), ...]
track_history = defaultdict(list)
track_jersey_colors = defaultdict(list)  # 各IDのユニフォーム色サンプル

def torso_crop(frame, box):
    """bboxの上半身中央（ユニフォームが写る領域）を切り出す"""
    x1, y1, x2, y2 = map(int, box)
    bw, bh = x2 - x1, y2 - y1
    # 上半身の中央あたりを狙う（頭と脚を避ける）
    cx1 = x1 + int(bw * 0.25); cx2 = x2 - int(bw * 0.25)
    cy1 = y1 + int(bh * 0.15); cy2 = y1 + int(bh * 0.50)
    cx1, cy1 = max(cx1, 0), max(cy1, 0)
    crop = frame[cy1:cy2, cx1:cx2]
    return crop

def dominant_color(crop):
    """切り出し領域の代表色をHSVで返す（緑のコートや白背景を軽く除外）"""
    if crop is None or crop.size == 0:
        return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).reshape(-1, 3).astype(np.float32)
    # 彩度が極端に低い（白/灰）や明度が低い（影）画素を軽く除外
    mask = (hsv[:, 1] > 40) & (hsv[:, 2] > 40)
    sel = hsv[mask] if mask.sum() > 20 else hsv
    return sel.mean(axis=0)  # [H, S, V]

# track() はジェネレータでフレームを1枚ずつ返す
results = det_model.track(
    source=VIDEO_PATH, stream=True, persist=True,
    classes=[PERSON_CLASS], conf=CONF, tracker="bytetrack.yaml",
    device=DEVICE, verbose=False,
)

frames = []  # 後で描画に使うため元フレームを保持（長い動画では省メモリ化を検討）
for f_idx, r in enumerate(results):
    frame = r.orig_img
    frames.append(frame)
    if r.boxes is None or r.boxes.id is None:
        continue
    boxes = r.boxes.xyxy.cpu().numpy()
    ids = r.boxes.id.cpu().numpy().astype(int)
    for box, tid in zip(boxes, ids):
        track_history[tid].append((f_idx, *box.tolist()))
        col = dominant_color(torso_crop(frame, box))
        if col is not None:
            track_jersey_colors[tid].append(col)

print(f"✅ 追跡完了: {len(frames)}フレーム / 検出IDユニーク数 = {len(track_history)}")


## 4. チーム分け（ユニフォーム色でクラスタリング）

各選手IDの代表的なユニフォーム色を求め、KMeans で **2チーム** に分けます。
色相（Hue）を円環として扱うため sin/cos に展開してからクラスタリングします。
（審判やベンチなど少数の外れ値は、出現フレーム数でフィルタして除外できます）


In [ ]:
from sklearn.cluster import KMeans

# 出現回数が少ないID（誤検出・審判など）は除外
MIN_FRAMES = max(3, len(frames) // 40)
valid_ids = [tid for tid, cols in track_jersey_colors.items()
             if len(cols) >= MIN_FRAMES]

# 各IDの代表色（中央値）を特徴量化: Hueは円環なのでsin/cos、S/Vは正規化
feats, feat_ids = [], []
for tid in valid_ids:
    arr = np.array(track_jersey_colors[tid])
    h_med = np.median(arr[:, 0]) * 2.0          # OpenCV Hは0-179 → 0-360度
    s_med = np.median(arr[:, 1]) / 255.0
    v_med = np.median(arr[:, 2]) / 255.0
    rad = np.deg2rad(h_med)
    feats.append([np.cos(rad) * s_med, np.sin(rad) * s_med, v_med])
    feat_ids.append(tid)

feats = np.array(feats)
team_of = {}
if len(feat_ids) >= 2:
    km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(feats)
    for tid, lab in zip(feat_ids, km.labels_):
        team_of[tid] = int(lab)
    print("✅ チーム分け完了")
    for t in (0, 1):
        members = [i for i in feat_ids if team_of[i] == t]
        print(f"  チーム{t}: {len(members)}人  (IDs: {members})")
else:
    print("⚠️ 選手が2人以上検出できませんでした。CONFやMIN_FRAMESを調整してください。")

# チームごとの描画色（BGR）
TEAM_COLORS = {0: (0, 0, 255), 1: (255, 128, 0)}  # 赤 / 青
UNKNOWN_COLOR = (160, 160, 160)                    # グレー


## 5. 色分けした注釈付き動画を書き出す

各フレームに、チーム色の枠とID・チーム名を描画して動画に保存します。

In [ ]:
# frame_idx -> [(track_id, x1,y1,x2,y2), ...] に整理し直す
by_frame = defaultdict(list)
for tid, dets in track_history.items():
    for (f_idx, x1, y1, x2, y2) in dets:
        by_frame[f_idx].append((tid, x1, y1, x2, y2))

out_path = "output_team_colored.mp4"
H, W = frames[0].shape[:2]
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

for f_idx, frame in enumerate(frames):
    canvas = frame.copy()
    for (tid, x1, y1, x2, y2) in by_frame.get(f_idx, []):
        team = team_of.get(tid)
        color = TEAM_COLORS.get(team, UNKNOWN_COLOR)
        p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
        cv2.rectangle(canvas, p1, p2, color, 2)
        label = f"ID{tid}" + (f" T{team}" if team is not None else "")
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(canvas, (p1[0], p1[1] - th - 6), (p1[0] + tw + 4, p1[1]), color, -1)
        cv2.putText(canvas, label, (p1[0] + 2, p1[1] - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    writer.write(canvas)

writer.release()
print("✅ 書き出し完了:", out_path)


In [ ]:
# Colab上でプレビュー再生（H.264に変換して埋め込み）
!ffmpeg -y -i output_team_colored.mp4 -vcodec libx264 -movflags +faststart preview.mp4 -loglevel error

from IPython.display import HTML
from base64 import b64encode
mp4 = open("preview.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width="640" controls><source src="{data_url}" type="video/mp4"></video>')


## 6. 個人の精密追跡（SAM2）

ここからは記事後半の「1人の選手を最後まで追う」パートです。
**SAM2（Segment Anything Model 2）** のビデオ予測器を使い、最初のフレームで指定した選手を
ピクセル単位のマスクで動画全体にわたって追跡します。

まず追跡したい選手を選びます（上で表示されたIDから1つ選んでください）。


In [ ]:
# 追跡したい選手のトラッキングID（上のチーム分け結果から選ぶ）
TARGET_ID = feat_ids[0] if feat_ids else None
print("追跡対象 ID:", TARGET_ID)

# そのIDが最初に登場するフレームとbboxを取得（SAM2への初期プロンプトに使う）
start_frame_idx, start_box = None, None
for (f_idx, x1, y1, x2, y2) in sorted(track_history[TARGET_ID]):
    start_frame_idx, start_box = f_idx, (x1, y1, x2, y2)
    break
print("開始フレーム:", start_frame_idx, "初期bbox:", start_box)


In [ ]:
# SAM2 のビデオ予測器で対象を追跡
# ultralytics 同梱の SAM2 を利用（重みは初回自動ダウンロード）
from ultralytics.models.sam import SAM2VideoPredictor

overrides = dict(conf=0.25, task="segment", mode="predict",
                 imgsz=1024, model="sam2.1_b.pt", device=DEVICE, verbose=False)
predictor = SAM2VideoPredictor(overrides=overrides)

# 初期フレームで bbox プロンプトを与え、以降のフレームを自動で追跡
x1, y1, x2, y2 = start_box
sam_results = predictor(source=VIDEO_PATH, bboxes=[[x1, y1, x2, y2]], labels=[1])

print("✅ SAM2 追跡完了。フレーム数:", len(sam_results))


In [ ]:
# SAM2のマスクを元動画に半透明で重ねて書き出す
sam_out = "output_sam2_track.mp4"
writer2 = cv2.VideoWriter(sam_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
overlay_color = np.array([0, 255, 255], dtype=np.uint8)  # 黄色

for i, res in enumerate(sam_results):
    base = frames[i].copy() if i < len(frames) else res.orig_img.copy()
    if res.masks is not None and len(res.masks) > 0:
        m = res.masks.data[0].cpu().numpy().astype(bool)
        if m.shape != base.shape[:2]:
            m = cv2.resize(m.astype(np.uint8), (base.shape[1], base.shape[0])) > 0
        base[m] = (0.5 * base[m] + 0.5 * overlay_color).astype(np.uint8)
    writer2.write(base)

writer2.release()
print("✅ 書き出し完了:", sam_out)


In [ ]:
# SAM2追跡結果のプレビュー
!ffmpeg -y -i output_sam2_track.mp4 -vcodec libx264 -movflags +faststart preview_sam2.mp4 -loglevel error
from IPython.display import HTML
from base64 import b64encode
mp4 = open("preview_sam2.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width="640" controls><source src="{data_url}" type="video/mp4"></video>')


## 7. 結果をダウンロード

In [ ]:
from google.colab import files
for f in ["output_team_colored.mp4", "output_sam2_track.mp4"]:
    if Path(f).exists():
        files.download(f)


---
## 📝 補足・チューニングのヒント

- **検出漏れ/誤検出が多い**: セル3の `CONF` を下げ下げ/上げする、`yolo11m.pt` → `yolo11x.pt` に変更
- **チーム分けが不安定**: セル4の `MIN_FRAMES` を上げて外れ値（審判・観客）を除外。ユニフォームが白×薄色など近い場合は `torso_crop` の切り出し範囲を調整
- **処理が重い**: 短いクリップにする、`yolo11n.pt` を使う、フレームを間引く
- **SAM2でIDが乗り移る**: 記事でも「1人を最後まで追う」はまだ発展途上。定期的に bbox プロンプトを与え直す（複数フレームでプロンプト追加）と安定します

### このノートブックがやっていること（記事との対応）
| 記事の要素 | 対応セル | 使用技術 |
|---|---|---|
| 選手を検出 | 3 | YOLO11 |
| 選手にIDを付けて追跡 | 3 | ByteTrack |
| チームごとに色分け | 4–5 | KMeans（ユニフォーム色クラスタリング） |
| 1人を精密に追跡 | 6 | SAM2（Segment Anything Model 2） |

コストは Colab 無料枠 + OSSモデルのみ = **0円** で再現できます。
